# BACKTESTING_272 — Stratégie Momentum Dual EqualWeight
 
**Univers** : STOXX Europe 600  
**Période** : 2016-12-30 → 2026-02-20  
**Benchmark** : ESTRON Index


## 1. Importation des Libraries <a id='1'></a>

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))

import pandas as pd
import numpy as np
from IPython.display import display


from backtesting import DataLoader, BacktestEngine, ReportingEngine


## 2. Paramètres du portefeuille <a id='1'></a>

In [2]:
INITIAL_CAPITAL = 1_000_000

TRANSACTION_COST = 0.0005

SELECTION_QUANTILE = 0.40

METHODS = ["equal_weight"]

"""
- SECTOR_MIN : nombre minimum de poids par secteur // pas obligatoire
- SECTOR_MAX : nombre maximum de poids par secteur // pas obligatoire
"""
SECTOR_MIN = None
SECTOR_MAX = None

print(f"Capital initial   : {INITIAL_CAPITAL:,.0f} €")
print(f"Frais transaction : {TRANSACTION_COST * 10_000:.1f} bps")
print(f"Sélection         : top/bottom {SELECTION_QUANTILE*100:.0f}%")
print(f"Méthodes          : {METHODS}")
print(f"Contraintes       : SECTOR_MIN={SECTOR_MIN}, SECTOR_MAX={SECTOR_MAX}")

Capital initial   : 1,000,000 €
Frais transaction : 5.0 bps
Sélection         : top/bottom 40%
Méthodes          : ['equal_weight']
Contraintes       : SECTOR_MIN=None, SECTOR_MAX=None



## 3. Chargement des données <a id='1'></a>

In [3]:
"""
- start_date : date de début du backtest (format "YYYY-MM-DD")
- end_date : date de fin du backtest (format "YYYY-MM-DD")
"""

loader = DataLoader(start_date="2016-12-30", end_date="2026-02-20")

"""
- initial_capital    : implémente le capital initinial du portefeuille
- transaction_cost   : implémente le coût de transaction par trade 
- selection_quantile : implémente le quantile utilisé pour sélectionner les positions long/short au travers des signaux
- sector_min         : implémente le poids minimum autorisé par secteur (contrainte de diversification)
- sector_max         : implémente le poids maximum autorisé par secteur (contrainte de diversification)
"""

engine = BacktestEngine(
    data_loader=loader,
    initial_capital=INITIAL_CAPITAL,
    transaction_cost=TRANSACTION_COST,
    selection_quantile=SELECTION_QUANTILE,
    sector_min=SECTOR_MIN,
    sector_max=SECTOR_MAX,
)

results = {}

print(f"\nDonnées chargées : {len(loader.rebalancing_dates)} dates de rebalancement")
print(f"Tickers disponibles : {loader.prices.shape[1]}")
print(f"Période des prix : {loader.prices.index[0].date()} → {loader.prices.index[-1].date()}")

[DataLoader] Chargement des données en cours...
[DataLoader] Prêt : 111 dates de rebalancement (2016-12-30 → 2026-02-20).

Données chargées : 111 dates de rebalancement
Tickers disponibles : 980
Période des prix : 2015-09-01 → 2026-02-20



## 4. Backtesting <a id='1'></a>

In [4]:
"""
- Choix : equal_weight
"""
METHOD = "equal_weight"

if METHOD in METHODS:
    print(f"{'='*60}")
    print(f"  Lancement : {METHOD}")
    print(f"{'='*60}")
    results[METHOD] = engine.run(METHOD)
    nav_final = results[METHOD].nav.iloc[-1]
    total_ret = nav_final / INITIAL_CAPITAL - 1
    print(f"\n  NAV finale : {nav_final:,.0f} €  |  Rendement : {total_ret:.2%}")
else:
    print(f"'{METHOD}' non sélectionné dans METHODS — cellule ignorée.")

  Lancement : equal_weight

[BacktestEngine] Démarrage — méthode : equal_weight
  [  1/111] 2016-12-30 NAV=   1,014,807€  Long=232  Short=232
  [  2/111] 2017-01-31 NAV=   1,001,537€  Long=230  Short=230
  [  3/111] 2017-02-28 NAV=   1,004,065€  Long=231  Short=231
  [  4/111] 2017-03-31 NAV=   1,011,921€  Long=230  Short=230
  [  5/111] 2017-04-28 NAV=   1,003,515€  Long=230  Short=230
  [  6/111] 2017-05-31 NAV=   1,015,297€  Long=229  Short=229
  [  7/111] 2017-06-30 NAV=   1,032,306€  Long=231  Short=231
  [  8/111] 2017-07-31 NAV=   1,056,938€  Long=231  Short=231
  [  9/111] 2017-08-31 NAV=   1,055,094€  Long=231  Short=231
  [ 10/111] 2017-09-29 NAV=   1,072,169€  Long=230  Short=230
  [ 11/111] 2017-10-31 NAV=   1,078,991€  Long=203  Short=203
  [ 12/111] 2017-11-30 NAV=   1,070,015€  Long=232  Short=232
  [ 13/111] 2017-12-29 NAV=   1,093,571€  Long=231  Short=231
  [ 14/111] 2018-01-31 NAV=   1,081,447€  Long=232  Short=232
  [ 15/111] 2018-02-28 NAV=   1,089,766€  Long=167  


## 5. Reporting <a id='1'></a>

In [5]:
METHOD_TO_REPORT = METHODS.pop(0) 

"""
- AS_OF_DATE       : date d’arrêté des métriques de performance (format "YYYY-MM-DD")
"""
AS_OF_DATE = "2026-01-30"

reporter = ReportingEngine(results[METHOD_TO_REPORT], loader)
as_of = pd.Timestamp(AS_OF_DATE)

In [6]:

metrics = reporter.compute_metrics(as_of)


def format_metric(v, key=None):
    """
    - si la valeur est "--", elle est retournée telle qu'une valeur manquante,
    - si la valeur est un float :
         si la métrique est un ratio (Sharpe, IR, Corrélation, etc.) -> format décimal
         sinon (rendements, vol, drawdown, TC%) -> format en %
    - sinon, la valeur est convertie en chaîne de caractères.
    """
    if v == "--":
        return "--"

    if isinstance(v, float):

        if key is not None and ("(€)" in key or "€" in key):
            return f"{v:,.2f}"


        ratio_keys = (
            "Sharpe", "Sortino", "Information Ratio",
            "Calmar", "Corrélation", "Tracking Error"
        )
        if key is not None and any(rk in key for rk in ratio_keys):
            return f"{v:.4f}"


        return f"{v*100:.2f}%"

    return str(v)


"""
- Serie pandas indéxée par le nom des indicateurs et datée à la date de reporting
"""

metrics_series = pd.Series({
    k: format_metric(v, k) for k, v in metrics.items()
}, name=str(as_of.date()))
metrics_series.index.name = "Métrique"

categories = {
    "Performance": [
        "Rendement cumulé (période)", "Rendement cumulé 1 an",
        "Rendement cumulé 2 ans", "Rendement cumulé YTD", "Rendement cumulé MTD",
        "Rendement annualisé (période)",
        "CAGR (période)", "CAGR 1 an", "CAGR 2 ans", "CAGR YTD", "CAGR MTD",
    ]}

for cat, keys in categories.items():
    print(f"\n{'─'*50}")
    print(f"  {cat}")
    print(f"{'─'*50}")
    sub = {k: metrics.get(k, "--") for k in keys}
    for k, v in sub.items():
        val = format_metric(v, k)
        print(f"  {k:<45} {val:>12}")


──────────────────────────────────────────────────
  Performance
──────────────────────────────────────────────────
  Rendement cumulé (période)                          51.74%
  Rendement cumulé 1 an                               12.88%
  Rendement cumulé 2 ans                              18.58%
  Rendement cumulé YTD                                 2.48%
  Rendement cumulé MTD                                 2.48%
  Rendement annualisé (période)                        4.61%
  CAGR (période)                                       4.12%
  CAGR 1 an                                           13.15%
  CAGR 2 ans                                           9.58%
  CAGR YTD                                            34.79%
  CAGR MTD                                            34.79%


In [7]:
categories = {
    "Risque & Ratios": [
        "Sharpe ratio (période)", "Sharpe ratio 1 an",
        "Sortino ratio (période)", "Sortino ratio 1 an",
        "Tracking Error (période)", "Tracking Error 1 an", "Tracking Error 3 ans",
        "Information Ratio (période)", "Information Ratio 1 an",
        "Information Ratio 2 ans",
        "Calmar ratio (période)",
        "VaR 95% (période)", "VaR 95% 1 an",
        "CVaR 95% (période)", "CVaR 95% 1 an",
        "Volatilité (période)", "Volatilité 1 an", "Volatilité 2 ans",
        "Max Drawdown (période)", "Max Drawdown 1 an",
        "Max Drawdown 2 ans", "Max Drawdown YTD", "Max Drawdown MTD",
    ]}


for cat, keys in categories.items():
    print(f"\n{'─'*50}")
    print(f"  {cat}")
    print(f"{'─'*50}")
    sub = {k: metrics.get(k, "--") for k in keys}
    for k, v in sub.items():
        val = format_metric(v, k)
        print(f"  {k:<45} {val:>20}")


──────────────────────────────────────────────────
  Risque & Ratios
──────────────────────────────────────────────────
  Sharpe ratio (période)                                      0.4342
  Sharpe ratio 1 an                                           1.2998
  Sortino ratio (période)                                     0.4781
  Sortino ratio 1 an                                          1.8204
  Tracking Error (période)                                    0.0990
  Tracking Error 1 an                                         0.0815
  Tracking Error 3 ans                                        0.0746
  Information Ratio (période)                                 0.4342
  Information Ratio 1 an                                      1.2998
  Information Ratio 2 ans                                     0.8878
  Calmar ratio (période)                                      0.2524
  VaR 95% (période)                                            0.92%
  VaR 95% 1 an                                     

In [8]:

categories = {
       "Benchmark (ESTER)": [
        "Corrélation avec benchmark (période)",
        "Benchmark - Rendement cumulé (période)",
        "Benchmark - Rendement cumulé 1 an",
        "Benchmark - Rendement cumulé 2 ans",
        "Benchmark - Rendement cumulé YTD",
        "Benchmark - Rendement cumulé MTD",
        "Benchmark - Rendement annualisé (période)",
    ],
    "Coûts de transaction": ["TC total (€)", "TC total (%)"]}

for cat, keys in categories.items():
    print(f"\n{'─'*50}")
    print(f"  {cat}")
    print(f"{'─'*50}")
    sub = {k: metrics.get(k, "--") for k in keys}
    for k, v in sub.items():
        val = format_metric(v, k)
        print(f"  {k:<45} {val:>12}")


──────────────────────────────────────────────────
  Benchmark (ESTER)
──────────────────────────────────────────────────
  Corrélation avec benchmark (période)               -0.0056
  Benchmark - Rendement cumulé (période)               8.38%
  Benchmark - Rendement cumulé 1 an                    2.15%
  Benchmark - Rendement cumulé 2 ans                   5.92%
  Benchmark - Rendement cumulé YTD                     0.16%
  Benchmark - Rendement cumulé MTD                     0.16%
  Benchmark - Rendement annualisé (période)            1.25%

──────────────────────────────────────────────────
  Coûts de transaction
──────────────────────────────────────────────────
  TC total (€)                                     53,034.81
  TC total (%)                                         5.30%


In [9]:
all_metrics_df = reporter.compute_all_metrics()
display(all_metrics_df.tail(5))

,Rendement cumulé (période),Rendement cumulé 1 an,Rendement cumulé 2 ans,Rendement cumulé YTD,Rendement cumulé MTD,Rendement annualisé (période),CAGR (période),CAGR 1 an,CAGR 2 ans,CAGR YTD,...,Corrélation avec benchmark (période),Benchmark - Rendement cumulé (période),Benchmark - Rendement cumulé 1 an,Benchmark - Rendement cumulé 2 ans,Benchmark - Rendement cumulé YTD,Benchmark - Rendement cumulé MTD,Benchmark - Rendement annualisé (période),TC total (€),TC total (%),PnL (€)
Date,,,,,,,,,,,,,,,,,,,,,
2025-10-31,0.430809,0.069715,0.100596,0.066189,0.003118,0.040589,0.035813,0.071081,0.05191,0.080046,...,-0.008134,0.078571,0.0244,0.064479,0.019022,0.00166,0.01222,50329.71345,0.05033,372751.693863
2025-11-28,0.451504,0.078429,0.119692,0.08137,0.014239,0.041873,0.037107,0.079467,0.065161,0.089876,...,-0.007511,0.080191,0.023416,0.062846,0.020552,0.001501,0.012304,50638.371119,0.050638,392297.891623
2025-12-31,0.479069,0.101226,0.188314,0.101226,0.018362,0.043624,0.038824,0.100131,0.093329,0.101299,...,-0.006714,0.082104,0.022359,0.061158,0.022359,0.001771,0.012421,51513.559421,0.051514,417863.333787
2026-01-30,0.517372,0.128836,0.185818,0.024824,0.024824,0.04611,0.041224,0.131543,0.095782,0.347899,...,-0.005595,0.083847,0.021518,0.059186,0.001611,0.001611,0.012511,53034.814233,0.053035,453060.961864
2026-02-20,0.542183,0.123311,0.195861,0.04134,0.016116,0.047633,0.042744,0.124394,0.099586,0.336579,...,-0.004862,0.085069,0.021045,0.057965,0.00274,0.001127,0.012569,53376.482928,0.053376,476478.30103



## 5. Graphiques <a id='1'></a>

### 5a. Rendements cumulés — Portefeuille vs Benchmark

In [10]:
fig_cumul = reporter.plot_cumulative_returns()
fig_cumul.show()

### 5b. Drawdowns du portefeuille

In [11]:
fig_dd = reporter.plot_drawdowns()
fig_dd.show()

### 5c. Volatilité glissante annualisée

In [12]:
fig_vol = reporter.plot_historical_volatility()
fig_vol.show()

### 5d. Corrélation glissante avec le Benchmark

In [13]:
fig_corr = reporter.plot_historical_correlation()
fig_corr.show()

### 5e. Composition du portefeuille

In [14]:

comp_date = pd.Timestamp(AS_OF_DATE)


comp_df = reporter.get_portfolio_composition(comp_date)
print(f"Portefeuille au {comp_date.date()} : {len(comp_df)} positions")
if comp_df.empty or 'Weight' not in comp_df.columns:
    print("  Composition indisponible pour cette date (probablement hors rebalancement).")
else:
    print(f"  Long  : {(comp_df['Weight'] > 0).sum()} positions")
    print(f"  Short : {(comp_df['Weight'] < 0).sum()} positions")
    print()
    display(comp_df.head())


Portefeuille au 2026-01-30 : 470 positions
  Long  : 235 positions
  Short : 235 positions



,Ticker,Name,Country,Currency,Sector,Industry,Price,Weight
0,JDEP NA Equity,JDE PEET'S NV,NETHERLANDS,EUR,Consommation de base,Produits alimentaires,31.62,0.004255
1,BARN SE Equity,BARRY CALLEBAUT AG-REG,SWITZERLAND,CHF,Consommation de base,Produits alimentaires,1352.00,0.004255
2,SALM NO Equity,SALMAR ASA,NORWAY,NOK,Consommation de base,Produits alimentaires,573.00,0.004255
3,BATS LN Equity,BRITISH AMERICAN TOBACCO PLC,BRITAIN,GBp,Consommation de base,Tabac,4376.00,0.004255
4,CCH LN Equity,COCA-COLA HBC AG-DI,SWITZERLAND,GBp,Consommation de base,Boissons,3950.00,0.004255


In [ ]:
comp_charts = reporter.plot_composition_barcharts(comp_date)

comp_charts["Sector"].show()

"""
--> problème dans l'ouput, après avoir analysé le script + trouvé une alternantive avec comp_df (juste au dessus)
qui donne les valeurs présentes en portefeuille à la date de reporting avec leurs informations,
on obtient quand même zéro. On peut penser que ca vient du trop faible poids de chaque valeur couplé au faible nombre de valeur
par secteur.

Script test alternaative : 

grouped = (comp_df.dropna(subset=["Sector"]).groupby("Sector")["Weight"].sum().sort_values()) * 100

print("Nb secteurs affichés:", grouped.shape[0])
grouped

"""

In [16]:
comp_charts["Country"].show()

In [17]:
comp_charts["Industry"].show()

In [18]:
comp_charts["Currency"].show()

## 6. Analyse du portefeuille

### 6a. Top 10 positions 

In [19]:
top10_weights = reporter.get_top_10_weights(comp_date)

print(f"Top 10 positions au {comp_date.date()} :")
display(top10_weights)

Top 10 positions au 2026-01-30 :


,Ticker,Name,Sector,Country,Weight
0,JDEP NA Equity,JDE PEET'S NV,Consommation de base,NETHERLANDS,0.004255
1,BARN SE Equity,BARRY CALLEBAUT AG-REG,Consommation de base,SWITZERLAND,0.004255
2,SALM NO Equity,SALMAR ASA,Consommation de base,NORWAY,0.004255
3,BATS LN Equity,BRITISH AMERICAN TOBACCO PLC,Consommation de base,BRITAIN,0.004255
4,CCH LN Equity,COCA-COLA HBC AG-DI,Consommation de base,SWITZERLAND,0.004255
5,SBRY LN Equity,SAINSBURY (J) PLC,Consommation de base,BRITAIN,0.004255
6,MOWI NO Equity,MOWI ASA,Consommation de base,NORWAY,0.004255
7,RBREW DC Equity,ROYAL UNIBREW,Consommation de base,DENMARK,0.004255
8,IMB LN Equity,IMPERIAL BRANDS PLC,Consommation de base,BRITAIN,0.004255
9,TSCO LN Equity,TESCO PLC,Consommation de base,BRITAIN,0.004255


### 6b. Top 10 contributions à la performance

In [20]:
top5_pos, top5_neg = reporter.get_top_10_return_contribution(comp_date)

print(f"Top 5 contributeurs POSITIFS (période {comp_date.date()}) :")
display(top5_pos)

print(f"Top 5 contributeurs NÉGATIFS (période {comp_date.date()}) :")
display(top5_neg)

Top 5 contributeurs POSITIFS (période 2026-01-30) :


,Ticker,Contribution,Name,Sector
0,ASML NA Equity,0.002826,ASML HOLDING NV,Technologies de l'information
1,SBMO NA Equity,0.002066,SBM OFFSHORE NV,Energie
2,BESI NA Equity,0.002035,BE SEMICONDUCTOR INDUSTRIES,Technologies de l'information
3,EXPN LN Equity,0.001592,EXPERIAN PLC,Industrie
4,MT NA Equity,0.001560,ARCELORMITTAL,Matériaux


Top 5 contributeurs NÉGATIFS (période 2026-01-30) :


,Ticker,Contribution,Name,Sector
0,ASM NA Equity,-0.003286,ASM INTERNATIONAL NV,Technologies de l'information
1,BEZ LN Equity,-0.003202,BEAZLEY PLC,Finance
2,LOTB BB Equity,-0.002311,LOTUS BAKERIES,Consommation de base
3,INPST NA Equity,-0.002307,INPOST SA,Industrie
4,DIE BB Equity,-0.002243,D'IETEREN GROUP,Consommation discrétionnaire


### 6c. Top 10 contributions au risque du portefeuille (MCTR)

In [21]:
top5_risk_pos, top5_risk_neg = reporter.get_top_10_risk_contribution(comp_date)

print(f"Top 5 contributeurs POSITIFS au risque au {comp_date.date()} :")
display(top5_risk_pos)

print(f"Top 5 contributeurs NÉGATIFS au risque au {comp_date.date()} :")
display(top5_risk_neg)

Top 5 contributeurs POSITIFS au risque au 2026-01-30 :


,Ticker,Risk Contribution,Weight,Name,Sector
0,ABVX FP Equity,0.005589,0.004255,ABIVAX SA,Soins de santé
1,R3NK GY Equity,0.001051,0.004255,RENK GROUP AG,Industrie
2,HAG GY Equity,0.000987,0.004255,HENSOLDT AG,Industrie
3,TKA GY Equity,0.000902,0.004255,THYSSENKRUPP AG,Matériaux
4,ENR GY Equity,0.000890,0.004255,SIEMENS ENERGY AG,Industrie


Top 5 contributeurs NÉGATIFS au risque au 2026-01-30 :


,Ticker,Risk Contribution,Weight,Name,Sector
0,SPL PW Equity,-0.000402,-0.004255,SANTANDER BANK POLSKA SA,Finance
1,KOG NO Equity,-0.000388,-0.004255,KONGSBERG GRUPPEN ASA,Industrie
2,BNP FP Equity,-0.000366,-0.004255,BNP PARIBAS,Finance
3,MTX GY Equity,-0.000356,-0.004255,MTU AERO ENGINES AG,Industrie
4,SQN SE Equity,-0.000299,-0.004255,SWISSQUOTE GROUP HOLDING-REG,Finance


### 6d. Rendements cumulés calendaires du portefeuille

In [22]:
fig_calendar_returns = reporter.plot_calendar_returns_heatmap()

fig_calendar_returns.show()


## 7. Portfolio PnL

PnL cumulé du portefeuille

In [23]:
fig_pnl = reporter.plot_pnl()
fig_pnl.show()

## Exportation des Résultats

In [24]:
out_path = reporter.run_full_report(output_dir=None)
print("Reporting folder:", out_path)

out_dir = Path(out_path)

nav_files = sorted(out_dir.glob("nav_MomentumDual_*.parquet"))

metrics_files = sorted(out_dir.glob("metriques_MomentumDual_*.parquet"))

comp_files = sorted(out_dir.glob("composition_detaillee_MomentumDual_*.parquet"))

bbu_files = sorted(out_dir.glob("bbu_MomentumDual_*.csv"))


[ReportingEngine] Génération du reporting dans : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_EqualWeight_20161230_20260220


[Report] Métriques exportées : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_EqualWeight_20161230_20260220/metriques_MomentumDual_EqualWeight.parquet
[Report] BBU CSV exporté : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_EqualWeight_20161230_20260220/bbu_MomentumDual_EqualWeight.csv
[Report] Composition détaillée exportée : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_EqualWeight_20161230_20260220/composition_detaillee_MomentumDual_EqualWeight.parquet
[Report] NAV exportée : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_EqualWeight_20161230_20260220/nav_MomentumDual_EqualWeight.parquet
[ReportingEngine] Reporting complet généré dans : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_EqualWeight_20161230_20260220

Reporting folder: /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_EqualWeight_20161230_20260220


In [25]:

comp_files = sorted(out_dir.glob("composition_detaillee_MomentumDual_*.parquet"))
comp_df = pd.read_parquet(comp_files[0])
comp_df = pd.DataFrame(comp_df) 
comp_df["Date"] = pd.to_datetime(comp_df["Date"]) 
comp_files_df = comp_df[comp_df["Date"] == pd.Timestamp(AS_OF_DATE)] 
display(comp_files_df)

,Date,Ticker,Name,Country,Currency,Sector,Industry,Price,Weight
47280,2026-01-30,JDEP NA Equity,JDE PEET'S NV,NETHERLANDS,EUR,Consommation de base,Produits alimentaires,31.62,0.004255
47281,2026-01-30,BARN SE Equity,BARRY CALLEBAUT AG-REG,SWITZERLAND,CHF,Consommation de base,Produits alimentaires,1352.00,0.004255
47282,2026-01-30,SALM NO Equity,SALMAR ASA,NORWAY,NOK,Consommation de base,Produits alimentaires,573.00,0.004255
47283,2026-01-30,BATS LN Equity,BRITISH AMERICAN TOBACCO PLC,BRITAIN,GBp,Consommation de base,Tabac,4376.00,0.004255
47284,2026-01-30,CCH LN Equity,COCA-COLA HBC AG-DI,SWITZERLAND,GBp,Consommation de base,Boissons,3950.00,0.004255
...,...,...,...,...,...,...,...,...,...
47745,2026-01-30,SOP FP Equity,SOPRA STERIA GROUP,FRANCE,EUR,Technologies de l'information,Services IT,154.20,-0.004255
47746,2026-01-30,SAP GY Equity,SAP SE,GERMANY,EUR,Technologies de l'information,Logiciels,170.56,-0.004255
47747,2026-01-30,REY IM Equity,REPLY SPA,ITALY,EUR,Technologies de l'information,Services IT,110.40,-0.004255
47748,2026-01-30,NEM GY Equity,NEMETSCHEK SE,GERMANY,EUR,Technologies de l'information,Logiciels,73.95,-0.004255


In [26]:
bbu_df = pd.read_csv(bbu_files[0])
display(bbu_df.head())

,Date,Ticker,Weights
0,2016-12-30,KESKOB FH Equity,0.00431
1,2016-12-30,MRW LN Equity,0.00431
2,2016-12-30,TSCO LN Equity,0.00431
3,2016-12-30,MOWI NO Equity,0.00431
4,2016-12-30,JMT PL Equity,0.00431
